# Cell 1 — Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Cell 2 — Device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Cell 3 — Load Dataset

In [3]:
with open("../data/processed/dl/dl_train_test_split.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

print(X_train.shape)
print(X_test.shape)

(1539641, 103)
(384911, 103)


# Cell 4 — Feature Scaling (Required by Dataset Pipeline)

Scaling is part of the dataset preprocessing workflow used for ML and DL experiments.

In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Cell 5 — Encode Labels

In [5]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)
print("Classes:", num_classes)

Classes: 15


# Cell 6 — Reshape Data for LSTM

LSTM expects:

```
(batch, sequence_length, features)
```

For tabular data we treat **each feature as a timestep**.

In [6]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(X_train.shape)

(1539641, 103, 1)


# Cell 7 — Convert to Tensors

In [7]:
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train = torch.tensor(y_train, dtype=torch.long).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)

# Cell 8 — DataLoader

In [8]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

# Cell 9 — LSTM Model

Typical IDS LSTM architecture.

In [9]:
class LSTMModel(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers, num_classes):

        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]  # last timestep

        out = self.fc(out)

        return out

# Cell 10 — Initialize Model

In [10]:
input_size = 1
hidden_size = 64
num_layers = 2

model = LSTMModel(
    input_size,
    hidden_size,
    num_layers,
    num_classes
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# Cell 11 — Training Loop

In [11]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss}")

Epoch 1/10 Loss: 2468.4916746914387
Epoch 2/10 Loss: 1551.791012018919
Epoch 3/10 Loss: 917.8106763884425
Epoch 4/10 Loss: 853.4587019793689
Epoch 5/10 Loss: 740.1359452791512
Epoch 6/10 Loss: 682.0634355992079
Epoch 7/10 Loss: 659.3576552662998
Epoch 8/10 Loss: 647.4481028169394
Epoch 9/10 Loss: 633.1641721762717
Epoch 10/10 Loss: 631.3856875412166


# Cell 12 — Evaluation

In [14]:
model.eval()

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

all_preds = []
all_true = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_true.extend(y_batch.cpu().numpy())

import numpy as np
y_pred = np.array(all_preds)
y_true = np.array(all_true)


# Cell 13 — Accuracy

In [15]:
accuracy = accuracy_score(y_true, y_pred)

print("LSTM Accuracy:", accuracy)

LSTM Accuracy: 0.9508639659557664


# Cell 14 — Classification Report

In [16]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.95      0.96      4806
           1       0.92      0.67      0.78      9871
           2       0.99      1.00      1.00     13588
           3       0.99      1.00      0.99     10013
           4       1.00      1.00      1.00     24314
           5       0.00      0.00      0.00       171
           6       1.00      1.00      1.00        72
           7       1.00      1.00      1.00    275611
           8       0.59      0.31      0.40      9987
           9       0.97      0.99      0.98      3996
          10       0.99      0.97      0.98      1938
          11       0.44      0.77      0.56     10165
          12       0.57      0.47      0.52      7361
          13       0.97      0.85      0.90     10005
          14       0.49      0.85      0.62      3013

    accuracy                           0.95    384911
   macro avg       0.79      0.79      0.78    384911
weighted avg       0.96   

e:\LLM Threat Detection on IIOT\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\LLM Threat Detection on IIOT\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\LLM Threat Detection on IIOT\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

# Cell 15 — Save Model

In [17]:
torch.save(
    model.state_dict(),
    "../models/dl/lstm_model.pth"
)